# Dataset Preparation: Narrative + QA

This notebook downloads and preprocesses two types of data:
1. **Narrative** — stories from `HuggingFaceTB/cosmopedia` (stories split)
2. **QA with context** — from `rajpurkar/squad` + `rajpurkar/squad_v2`

All data is unified into a single schema and saved as a HuggingFace dataset with 90/10 train/validation splits.

## 0. Install dependencies

In [1]:
!pip install -q datasets

You should consider upgrading via the '/Users/kirill/My Folder/study/thesis/experiments/venv/bin/python3 -m pip install --upgrade pip' command.


## 1. Configuration

In [4]:
# ── Sampling parameters ──────────────────────────────────────────────
# Set how many samples to keep from each source BEFORE merging.
# Set to None to keep everything (warning: Cosmopedia stories can be 1M+).

NARRATIVE_SAMPLE_SIZE = 600_000   # number of Cosmopedia story samples to keep
QA_SAMPLE_SIZE        = 200_000   # number of QA samples to keep (drawn from SQuAD 1.1 + 2.0 combined)

# ── Train / Validation split ─────────────────────────────────────────
VAL_RATIO = 0.10                  # 10% validation

# ── Output ────────────────────────────────────────────────────────────
OUTPUT_DIR = "data/unified_dataset"  # where to save the final HF dataset

# ── Reproducibility ──────────────────────────────────────────────────
SEED = 42

## 2. Load raw datasets

In [2]:
from datasets import load_dataset, concatenate_datasets

# ── 2a. Narrative: Cosmopedia stories ────────────────────────────────
print("Loading Cosmopedia (stories split)...")
cosmo = load_dataset(
    "HuggingFaceTB/cosmopedia",
    "stories",               # only the stories subset
    split="train",
)
print(f"  Loaded {len(cosmo):,} story samples")
print(f"  Columns: {cosmo.column_names}")
print(f"  Example keys: {list(cosmo[0].keys())}")
print()

/Users/kirill/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/kirill/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Cosmopedia (stories split)...


KeyboardInterrupt: 

In [3]:
# ── 2b. QA: SQuAD v1.1 + v2.0 ──────────────────────────────────────
print("Loading SQuAD v1.1...")
squad1_train = load_dataset("rajpurkar/squad", split="train")
squad1_val   = load_dataset("rajpurkar/squad", split="validation")
squad1 = concatenate_datasets([squad1_train, squad1_val])
print(f"  SQuAD v1.1 total: {len(squad1):,}")

print("Loading SQuAD v2.0...")
squad2_train = load_dataset("rajpurkar/squad_v2", split="train")
squad2_val   = load_dataset("rajpurkar/squad_v2", split="validation")
squad2 = concatenate_datasets([squad2_train, squad2_val])
print(f"  SQuAD v2.0 total: {len(squad2):,}")

# Combine both
squad_all = concatenate_datasets([squad1, squad2])
print(f"  Combined SQuAD total: {len(squad_all):,}")
print(f"  Columns: {squad_all.column_names}")
print()

Loading SQuAD v1.1...


Generating validation split: 100%|██████████| 10570/10570 [00:00<00:00, 1282806.52 examples/s]


  SQuAD v1.1 total: 98,169
Loading SQuAD v2.0...


Generating validation split: 100%|██████████| 11873/11873 [00:00<00:00, 1628641.51 examples/s]


  SQuAD v2.0 total: 142,192
  Combined SQuAD total: 240,361
  Columns: ['id', 'title', 'context', 'question', 'answers']



## 3. Quick peek at raw data

In [4]:
print("=" * 60)
print("COSMOPEDIA STORY SAMPLE")
print("=" * 60)
sample = cosmo[0]
for k, v in sample.items():
    preview = str(v)[:200] if isinstance(v, str) else v
    print(f"  {k}: {preview}")

print()
print("=" * 60)
print("SQUAD SAMPLE")
print("=" * 60)
sample = squad_all[0]
for k, v in sample.items():
    preview = str(v)[:200] if isinstance(v, str) else v
    print(f"  {k}: {preview}")

COSMOPEDIA STORY SAMPLE
  text:  Once upon a time, in a village called Kiwiland, there lived two best friends named Kiwi and Koala. They loved exploring the world around them and learning new things every day! One day, they stumbled
  prompt: Write an educational story (3-5 paragraphs) targeted at young children using simple words. The story should be inspired from this text snippet: 
“How do cultural beliefs and values influence decision 
  text_token_length: 520
  seed_data: ultrachat
  format: story_children
  audience: young_children

SQUAD SAMPLE
  id: 5733be284776f41900661182
  title: University_of_Notre_Dame
  context: Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper sta
  question: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
  answers: {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


## 4. Subsample to target sizes

In [5]:
if NARRATIVE_SAMPLE_SIZE is not None and len(cosmo) > NARRATIVE_SAMPLE_SIZE:
    cosmo = cosmo.shuffle(seed=SEED).select(range(NARRATIVE_SAMPLE_SIZE))
    print(f"Narrative subsampled to {len(cosmo):,}")
else:
    print(f"Narrative: keeping all {len(cosmo):,} samples")

# For QA — filter out unanswerable questions from SQuAD v2 (empty answer text)
# These have answers.text == [] which is not useful for training "answer from context"
squad_all = squad_all.filter(
    lambda x: len(x["answers"]["text"]) > 0,
    desc="Filtering out unanswerable questions",
)
print(f"QA after removing unanswerable: {len(squad_all):,}")

if QA_SAMPLE_SIZE is not None and len(squad_all) > QA_SAMPLE_SIZE:
    squad_all = squad_all.shuffle(seed=SEED).select(range(QA_SAMPLE_SIZE))
    print(f"QA subsampled to {len(squad_all):,}")
else:
    print(f"QA: keeping all {len(squad_all):,} samples")

Narrative subsampled to 600,000


Filtering out unanswerable questions: 100%|██████████| 240361/240361 [00:01<00:00, 156727.15 examples/s]

QA after removing unanswerable: 190,918
QA: keeping all 190,918 samples


## 5. Transform into unified schema

Target schema:
```
id          str  — unique example identifier
split       str  — "train" | "validation"  (assigned later)
task        str  — "narrative" | "qa"
source_text str  — text fed to the embedder (BERT)
question    str  — for qa: the question; for narrative: ""
answer      str  — for qa: the answer text; for narrative: same as source_text
```

In [6]:
from datasets import Dataset


def transform_narrative(dataset):
    """Convert Cosmopedia stories to unified schema."""
    records = []
    # Cosmopedia stories have a 'text' column with the full story
    text_col = "text" if "text" in dataset.column_names else dataset.column_names[0]

    for i, row in enumerate(dataset):
        text = row[text_col].strip()
        if not text:
            continue
        records.append({
            "id":          f"narrative_{i:07d}",
            "split":       "",           # placeholder — assigned in step 6
            "task":        "narrative",
            "source_text": text,
            "question":    "",
            "answer":      text,
        })
    return Dataset.from_list(records)


def transform_qa(dataset):
    """Convert SQuAD-format data to unified schema."""
    records = []
    for i, row in enumerate(dataset):
        context  = row["context"].strip()
        question = row["question"].strip()
        answer   = row["answers"]["text"][0].strip()

        if not context or not question or not answer:
            continue

        records.append({
            "id":          f"qa_{i:07d}",
            "split":       "",           # placeholder
            "task":        "qa",
            "source_text": context,
            "question":    question,
            "answer":      answer,
        })
    return Dataset.from_list(records)


print("Transforming narrative data...")
narrative_ds = transform_narrative(cosmo)
print(f"  → {len(narrative_ds):,} narrative samples")

print("Transforming QA data...")
qa_ds = transform_qa(squad_all)
print(f"  → {len(qa_ds):,} QA samples")

Transforming narrative data...
  → 600,000 narrative samples
Transforming QA data...
  → 190,918 QA samples


## 6. Merge & create train/validation splits

In [7]:
from datasets import DatasetDict

# Merge both task datasets
full_ds = concatenate_datasets([narrative_ds, qa_ds])
print(f"Total merged dataset: {len(full_ds):,} samples")

# Shuffle and split 90/10
full_ds = full_ds.shuffle(seed=SEED)
split = full_ds.train_test_split(test_size=VAL_RATIO, seed=SEED)

train_ds = split["train"]
val_ds   = split["test"]


# Assign the 'split' column
def set_split(example, split_name):
    example["split"] = split_name
    return example


train_ds = train_ds.map(lambda x: set_split(x, "train"), desc="Setting split=train")
val_ds   = val_ds.map(lambda x: set_split(x, "validation"), desc="Setting split=validation")

dataset_dict = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
})

print(f"\nTrain:      {len(train_ds):,}")
print(f"Validation: {len(val_ds):,}")

Total merged dataset: 790,918 samples


Setting split=validation: 100%|██████████| 79092/79092 [00:02<00:00, 28592.96 examples/s]


Train:      711,826
Validation: 79,092


## 7. Verify schema & stats

In [8]:
import pandas as pd

print("Schema:")
print(dataset_dict["train"].features)
print()

# Task distribution per split
for split_name in ["train", "validation"]:
    ds = dataset_dict[split_name]
    df = pd.DataFrame({"task": ds["task"]})
    print(f"--- {split_name} ---")
    print(df["task"].value_counts().to_string())
    print()

Schema:
{'id': Value('string'), 'split': Value('string'), 'task': Value('string'), 'source_text': Value('string'), 'question': Value('string'), 'answer': Value('string')}

--- train ---
task
narrative    540128
qa           171698

--- validation ---
task
narrative    59872
qa           19220



In [9]:
# Spot-check examples
print("=" * 60)
print("NARRATIVE EXAMPLE (from train)")
print("=" * 60)
narr_examples = [x for x in dataset_dict["train"] if x["task"] == "narrative"]
ex = narr_examples[0]
print(f"ID:          {ex['id']}")
print(f"Task:        {ex['task']}")
print(f"Question:    '{ex['question']}'")
print(f"Source text:  {ex['source_text'][:300]}...")
print(f"Answer:       {ex['answer'][:300]}...")

print()
print("=" * 60)
print("QA EXAMPLE (from train)")
print("=" * 60)
qa_examples = [x for x in dataset_dict["train"] if x["task"] == "qa"]
ex = qa_examples[0]
print(f"ID:          {ex['id']}")
print(f"Task:        {ex['task']}")
print(f"Question:    {ex['question']}")
print(f"Source text:  {ex['source_text'][:300]}...")
print(f"Answer:       {ex['answer']}")

NARRATIVE EXAMPLE (from train)
ID:          narrative_0202626
Task:        narrative
Question:    ''
Source text:  I've always been fascinated by niche topics, especially those relating to true crime and serial killers. It might seem odd to some, but there's something about understanding the minds of these notorious figures that has intrigued me for years. So, when I decided to pursue my Master's degree in psych...
Answer:       I've always been fascinated by niche topics, especially those relating to true crime and serial killers. It might seem odd to some, but there's something about understanding the minds of these notorious figures that has intrigued me for years. So, when I decided to pursue my Master's degree in psych...

QA EXAMPLE (from train)
ID:          qa_0046184
Task:        qa
Question:    Who ruled Egypt in 1952?
Source text:  In 1951, the Conservative Party returned to power in Britain, under the leadership of Winston Churchill. Churchill and the Conservatives believed 

In [10]:
# Quick length stats
import numpy as np

for task_name in ["narrative", "qa"]:
    subset = [x for x in dataset_dict["train"] if x["task"] == task_name]
    lengths = [len(x["source_text"].split()) for x in subset[:5000]]  # sample for speed
    print(f"[{task_name}] source_text word count (sampled up to 5k):")
    print(f"  min={np.min(lengths)}, median={np.median(lengths):.0f}, "
          f"mean={np.mean(lengths):.0f}, max={np.max(lengths)}")
    if task_name == "qa":
        ans_lengths = [len(x["answer"].split()) for x in subset[:5000]]
        print(f"  answer word count: min={np.min(ans_lengths)}, "
              f"median={np.median(ans_lengths):.0f}, mean={np.mean(ans_lengths):.0f}, "
              f"max={np.max(ans_lengths)}")
    print()

[narrative] source_text word count (sampled up to 5k):
  min=16, median=419, mean=424, max=965

[qa] source_text word count (sampled up to 5k):
  min=20, median=109, mean=119, max=533
  answer word count: min=1, median=2, mean=3, max=27



## 8. Save to disk (HF format)

In [ ]:
# dataset_dict.save_to_disk(OUTPUT_DIR)
# print(f"Dataset saved to: {OUTPUT_DIR}")
# print()

# Verify it loads back
from datasets import load_from_disk
reloaded = load_from_disk(OUTPUT_DIR)
print("Reloaded successfully:")
print(reloaded)

Reloaded successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'split', 'task', 'source_text', 'question', 'answer'],
        num_rows: 711826
    })
    validation: Dataset({
        features: ['id', 'split', 'task', 'source_text', 'question', 'answer'],
        num_rows: 79092
    })
})


In [13]:
from IPython.display import display, HTML
display(HTML("""
<style>
.jp-OutputArea-output pre {
    white-space: pre-wrap !important;
    word-wrap: break-word !important;
    overflow-x: hidden !important;
}
</style>
"""))

In [24]:
import pprint
pp = pprint.PrettyPrinter(width=150)
pp.pprint(reloaded['train'][0])

{'answer': 'Gamal Abdul Nasser',
 'id': 'qa_0046184',
 'question': 'Who ruled Egypt in 1952?',
 'source_text': 'In 1951, the Conservative Party returned to power in Britain, under the leadership of Winston Churchill. Churchill and the '
                "Conservatives believed that Britain's position as a world power relied on the continued existence of the empire, with the base at "
                'the Suez Canal allowing Britain to maintain its pre-eminent position in the Middle East in spite of the loss of India. However, '
                "Churchill could not ignore Gamal Abdul Nasser's new revolutionary government of Egypt that had taken power in 1952, and the "
                'following year it was agreed that British troops would withdraw from the Suez Canal zone and that Sudan would be granted '
                'self-determination by 1955, with independence to follow. Sudan was granted independence on 1 January 1956.',
 'split': 'train',
 'task': 'qa'}


In [ ]:
https://your-durev.com/sub/Jc4DaiTGJGR3vwB3g9q8vp/info/

## 9. (Optional) Push to Hugging Face Hub

Uncomment and fill in your repo name to upload.

In [ ]:
# from huggingface_hub import login
# login()  # paste your HF token when prompted
#
# HUB_REPO = "your-username/your-dataset-name"
# dataset_dict.push_to_hub(HUB_REPO, private=True)
# print(f"Pushed to https://huggingface.co/datasets/{HUB_REPO}")

---
**Done!** The dataset at `./unified_dataset` is ready to load with:
```python
from datasets import load_from_disk
ds = load_from_disk("./unified_dataset")
train = ds["train"]
val   = ds["validation"]
```